## Understanding Output Parsers: JSON and CSV

In [1]:
%%capture
!pip install --force-reinstall --no-cache-dir tenacity==8.2.3 --user
!pip install "ibm-watsonx-ai==1.0.8" --user
!pip install "ibm-watson-machine-learning==1.0.367" --user
!pip install "langchain-ibm==0.1.7" --user
!pip install "langchain-community==0.2.10" --user
!pip install "langchain-experimental==0.0.62" --user
!pip install "langchainhub==0.1.18" --user
!pip install "langchain==0.2.11" --user
!pip install "pypdf==4.2.0" --user
!pip install "chromadb==0.4.24" --user

In [ ]:
import os
os._exit(00)

In [1]:
# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')
import os
os.environ['ANONYMIZED_TELEMETRY'] = 'False'

from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes
from ibm_watson_machine_learning.foundation_models.extensions.langchain import WatsonxLLM

In [2]:
model_id = 'ibm/granite-4-h-small' 

parameters = {
    GenParams.MAX_NEW_TOKENS: 256,  # this controls the maximum number of tokens in the generated output
    GenParams.TEMPERATURE: 0.2, # this randomness or creativity of the model's responses 
}

credentials = {
    "url": "https://us-south.ml.cloud.ibm.com"
    # "api_key": "your api key here"
    # uncomment above and fill in the API key when running locally
}

project_id = "skills-network"

model = ModelInference(
    model_id=model_id,
    params=parameters,
    credentials=credentials,
    project_id=project_id
)
#Adding a llama model, because something is wrong with granite ig
llama_model = ModelInference(
    model_id='meta-llama/llama-4-maverick-17b-128e-instruct-fp8',
    params=parameters,
    credentials=credentials,
    project_id=project_id
)

In [3]:
llama_llm = WatsonxLLM(model = model)
llama_llm2 = WatsonxLLM(model=llama_model) #added llama model, because granite wasn't working ig

In [4]:
from langchain_core.prompts import PromptTemplate

In [5]:
# Import the JsonOutputParser from langchain_core to convert LLM responses into structured JSON
from langchain_core.output_parsers import JsonOutputParser
# Import BaseModel and Field from langchain_core's pydantic_v1 module
from langchain_core.pydantic_v1 import BaseModel, Field

In [6]:
# Define your desired data structure.
class Joke(BaseModel):
    setup: str = Field(description="question to set up a joke")
    punchline: str = Field(description="answer to resolve the joke")

In [13]:
# And a query intended to prompt a language model to populate the data structure.
joke_query = "Tell me a joke."

# Set up a parser + inject instructions into the prompt template.
output_parser = JsonOutputParser(pydantic_object=Joke)

# Get the formatting instructions for the output parser
# This generates guidance text that tells the LLM how to format its response
format_instructions = output_parser.get_format_instructions()

# Create a prompt template that includes:
# 1. Instructions for the LLM to answer the user's query
# 2. Format instructions to ensure the LLM returns properly structured data
# 3. The actual user query placeholder
#We will use this one, as the prompt template given by default wasn't working:
# Create a stronger prompt template that forbids conversational fluff
prompt = PromptTemplate(
    template=(
        "System: You are a strict JSON generator. Do NOT include any introductory or "
        "conversational text like 'Here is a joke' or 'Sure, I can help'. Your entire output "
        "must consist ONLY of the JSON object requested below.\n\n"
        "Format Instructions:\n{format_instructions}\n\n"
        "User Query: {query}\n"
    ),
    input_variables=["query"],
    partial_variables={"format_instructions": format_instructions},
)
#This one was not working:
#prompt = PromptTemplate(
#    template="Answer the user query.\n{format_instructions}\n{query}\n",
#    input_variables=["query"],  # Dynamic variables that will be provided when invoking the chain
#    partial_variables={"format_instructions": format_instructions},  # Static variables set once when creating the prompt
#)

# Create a processing chain that:
# 1. Formats the prompt using the template
# 2. Sends the formatted prompt to the Llama LLM
# 3. Parses the LLM's response using the output parser to extract structured data
chain = prompt | llama_llm2 | output_parser

# Invoke the chain with a specific query about jokes
# This will:
# 1. Format the prompt with the joke query
# 2. Send it to Llama
# 3. Parse the response into the str{'name': 'joke', 'arguments': '{}'}ucture defined by your output parser
# 4. Return the structured result
chain.invoke({"query": joke_query})

{'setup': "Why don't scientists trust atoms?",
 'punchline': 'Because they make up everything.'}

In [8]:
## COMMA-SEPERATED LIST PARSER

In [14]:
# Import the CommaSeparatedListOutputParser to parse LLM responses into Python lists
from langchain.output_parsers import CommaSeparatedListOutputParser

# Create an instance of the parser that will convert comma-separated text into a Python list
output_parser = CommaSeparatedListOutputParser()

# Get formatting instructions that will tell the LLM how to structure its response
# These instructions explain to the LLM that it should return items in a comma-separated format
format_instructions = output_parser.get_format_instructions()

# Create a prompt template that:
# 1. Instructs the LLM to answer the user query
# 2. Includes format instructions so the LLM knows to respond with comma-separated values
# 3. Asks the LLM to list five items of the specified subject
prompt = PromptTemplate(
    template="Answer the user query. {format_instructions}\nList five {subject}.",
    input_variables=["subject"],  # This variable will be provided when the chain is invoked
    partial_variables={"format_instructions": format_instructions},  # This variable is set once when creating the prompt
)

# Build a processing chain that:
# 1. Takes the subject and formats it into the prompt template
# 2. Sends the formatted prompt to the Llama LLM
# 3. Parses the LLM's response into a Python list using the CommaSeparatedListOutputParser
chain = prompt | llama_llm2 | output_parser

# Invoke the processing chain with "ice cream flavors" as the subject
# This will:
# 1. Substitute "ice cream flavors" into the prompt template
# 2. Send the formatted prompt to the Llama LLM
# 3. Parse the LLM's comma-separated response into a Python list
chain.invoke({"subject": "ice cream flavors"})

['Vanilla',
 'Chocolate',
 'Strawberry',
 'Cookies and Cream',
 'Mint Chocolate Chip<|eom|>']